# Full Pipeline — Crochet Chart Recognition with YOLO-OBB

This notebook is a **single-file walkthrough** of the end-to-end pipeline that turns a photographed crochet chart into a structured, machine-readable scheme. It is meant to be read top-to-bottom: anyone reviewing the project should come away with a clear picture of *what* each step does, *why* it is there, and *which tools* power it.

## Pipeline at a glance

```
                ┌───────────────────────┐
                │  Step 1 — Synthetic   │  procedural drawing + PNG templates
                │  data generation      │  -> 300 train / 60 val 640×640 PNGs
                └──────────┬────────────┘     + YOLO-OBB labels (.txt) + data.yaml
                           │
                           ▼
                ┌───────────────────────┐
                │  Step 2 — Training    │  Ultralytics YOLOv8n-OBB
                │  (real + synthetic)   │  local MPS or Colab GPU, degrees=180
                └──────────┬────────────┘     -> runs/obb/.../weights/best.pt
                           │
                           ▼
                ┌───────────────────────┐
                │  Step 3 — Real-image  │  adaptive tiled inference
                │  inference & rendering│  -> rotated bounding boxes
                └───────────────────────┘     -> reconstructed SVG scheme
```

## Technological stack

| Layer | Tools |
|---|---|
| **Modeling** | [Ultralytics YOLOv8n-OBB](https://docs.ultralytics.com/tasks/obb/) — anchor-free, oriented-bounding-box detector. Nano variant keeps training feasible on a laptop while still benefiting from the OBB head (rotated stitches need rotation-aware boxes). |
| **DL framework** | PyTorch (CUDA on Colab, **MPS** on Apple Silicon). |
| **Computer vision** | OpenCV (`cv2`) for loading, drawing, morphology, augmentation, JPEG re-encoding. |
| **Numerics & plotting** | NumPy, Matplotlib (overlays + reconstructed SVG). |
| **Data format** | YOLO-OBB label format: `cls x1 y1 x2 y2 x3 y3 x4 y4` with all coords normalised to `[0, 1]`. Dataset is described by a `data.yaml` (train/val splits + class names). |
| **Annotation feedback loop** | Label Studio (Docker), with `RectangleLabels` + rotation, fed by an auto-generated `tasks.json`. |
| **Domain code** | The `crochet` package (`crochet.detection`, `crochet.config`, `crochet.rendering`) wraps the inference + rendering logic so the same algorithm can run from the notebook, the Streamlit app, and the MCP server. |

## The 9-class system

The detector predicts oriented boxes for nine stitch classes, defined once in `crochet/config.py` and reused everywhere:

| ID | Name | Abbr | Hex colour | Role |
|---:|---|---|---|---|
| 0 | chain | ch | `#0000FF` | foundation chain (oval) |
| 1 | double | dc | `#FF00FF` | double crochet (T-stem with one slash) |
| 2 | double treble | dtr | `#00AA00` | very tall stitch (3 slashes) |
| 3 | enseble_chain | ec | `#FF8000` | vertical column of chain ovals |
| 4 | fan | fa | `#FF0000` | shell / fan (radiating spokes) |
| 5 | half_double | hd | `#00FFFF` | half-double crochet (T-stem, no slash) |
| 6 | noise | no | `#808080` | row numbers, arrows, annotations |
| 7 | single | sc | `#FFD700` | single crochet (cross) |
| 8 | treble | tr | `#00FF00` | treble crochet (2 slashes) |

## Setup — shared constants and paths

Everything below assumes the canonical project layout:

```
crochet_studio/
├── crochet/                    # importable package (config, detection, rendering, ...)
├── data/
│   ├── raw/easy/               # small clean test photos (single-shot inference)
│   ├── raw/big/                # large photos (require tiled inference)
│   └── raw/templates/          # PNG stitch symbols per class (used by Step 1)
├── notebooks/
│   ├── training_data/          # produced by Step 1
│   │   ├── train/{images,labels}/
│   │   ├── val/{images,labels}/
│   │   └── data.yaml
│   └── full_pipeline_YOLO_OBB.ipynb   # ← this notebook
└── runs/obb/obb_train*/weights/best.pt   # produced by Step 2
```

In [ ]:
# ── Common imports & constants ────────────────────────────────────────────
import os, sys, math, json, random, glob, shutil
from pathlib import Path
from collections import Counter
from dataclasses import dataclass

import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.transforms as transforms
import yaml

# Project root = parent of /notebooks. Adjust if you move the notebook.
PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) \
    if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")

# Make the `crochet` package importable for helpers further down.
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# 9-class system — ordered exactly as in data.yaml.
CLASS_MAP = {
    "chain":         0, "double":      1, "double treble": 2,
    "enseble_chain": 3, "fan":         4, "half_double":   5,
    "noise":         6, "single":      7, "treble":        8,
}
NUM_CLASSES = len(CLASS_MAP)
ID_TO_NAME = {v: k for k, v in CLASS_MAP.items()}

CLASS_CONFIG = {
    "chain":         {"abbr": "ch",  "color": "#0000FF"},
    "double":        {"abbr": "dc",  "color": "#FF00FF"},
    "double treble": {"abbr": "dtr", "color": "#00AA00"},
    "enseble_chain": {"abbr": "ec",  "color": "#FF8000"},
    "fan":           {"abbr": "fa",  "color": "#FF0000"},
    "half_double":   {"abbr": "hd",  "color": "#00FFFF"},
    "noise":         {"abbr": "no",  "color": "#808080"},
    "single":        {"abbr": "sc",  "color": "#FFD700"},
    "treble":        {"abbr": "tr",  "color": "#00FF00"},
}
DEFAULT_COLOR = "#808080"

# Inference defaults — same values are baked into crochet/config.py.
DEFAULT_CONF             = 0.25
DEFAULT_IOU              = 0.5
DEFAULT_TILE_SIZE        = 640
DEFAULT_OVERLAP          = 0.2
DEFAULT_TARGET_STITCH_PX = 100

print("Project dir:", PROJECT_DIR)
print("Classes    :", list(CLASS_MAP))

---
# Step 1 — Synthetic data generation

**Why synthetic data?** Hand-labelled real crochet charts are scarce and unbalanced (V-pattern doubles dominate every photo, while *enseble_chain*, *fan*, and *noise* are rare). We bridge that gap with a procedural generator that produces *thousands* of labelled charts cheaply and on demand. Real labelled photos are then **mixed in** during training so the model is exposed to genuine paper-and-pen artefacts.

**Strategy.** For each image the generator:
1. picks a colour palette that mimics real chart inks (green, purple, pink, black, blue);
2. picks a layout generator (V-pattern grid, expanding triangle, dense rows, mixed, fan grid, chain-only grid, single big fan) — weights are tuned so the dominant classes match real charts;
3. fills the canvas by either pasting a PNG template (`data/raw/templates/<cls>/*.png`) **or** calling a procedural drawer for that class — drawers exist for every class so we never run out of variety even when templates are missing;
4. records each pasted symbol's **rotated** four corners in YOLO-OBB normalised coords;
5. applies image-level augmentations (brightness/contrast, Gaussian blur, JPEG re-encoding, small global rotation, random scale-and-crop) so the network sees realistic degradations.

The result is a YOLO-ready dataset under `notebooks/training_data/` with a `data.yaml` describing the split and the 9 class names.

In [ ]:
# ── Configurable knobs ────────────────────────────────────────────────────
TEMPLATE_DIR = os.path.join(PROJECT_DIR, "data", "raw", "templates")
OUTPUT_DIR   = os.path.join(PROJECT_DIR, "notebooks", "training_data")

TRAIN_IMG = os.path.join(OUTPUT_DIR, "train", "images")
TRAIN_LBL = os.path.join(OUTPUT_DIR, "train", "labels")
VAL_IMG   = os.path.join(OUTPUT_DIR, "val",   "images")
VAL_LBL   = os.path.join(OUTPUT_DIR, "val",   "labels")
for d in [TRAIN_IMG, TRAIN_LBL, VAL_IMG, VAL_LBL]:
    os.makedirs(d, exist_ok=True)

NUM_TRAIN = 300
NUM_VAL   = 60
IMG_SIZE  = 640      # output image side (= YOLO input size)

# Hand-picked palettes that mimic real chart inks.
PALETTES = {
    "green":  {"fg": (40, 120, 60),  "bg": (245, 248, 240)},
    "purple": {"fg": (90, 50, 130),  "bg": (248, 244, 252)},
    "pink":   {"fg": (160, 50, 80),  "bg": (252, 245, 248)},
    "black":  {"fg": (30, 30, 30),   "bg": (255, 255, 255)},
    "blue":   {"fg": (50, 70, 150),  "bg": (245, 248, 255)},
}

### 1.1  Procedural symbol drawers

PNG templates alone cannot cover all nine classes (some classes have zero or one source PNGs). Each class therefore has a small **procedural drawer** that emits a transparent BGRA image on demand. The drawers are intentionally simple so the network learns the *geometry* of each glyph rather than memorising textures:

* `chain` — open ellipse
* `single` — `+` / `×` cross
* `double / treble / double treble` — vertical T-stem with 1, 2, or 3 diagonal slashes
* `half_double` — vertical T-stem with a single horizontal bar
* `fan` — fan of equally-spaced spokes from a base point
* `enseble_chain` — vertical stack of chain ovals
* `noise` — small circle, circled number, or arrow (mimics row counters and direction arrows on real charts)

In [ ]:
def draw_chain(size=40, thickness=2, color=(0, 0, 0)):
    w, h = size, int(size * 0.55)
    img = np.zeros((h + 4, w + 4, 4), dtype=np.uint8)
    cv.ellipse(img, ((w + 4) // 2, (h + 4) // 2),
               (w // 2, h // 2), 0, 0, 360, (*color, 255), thickness)
    return img

def draw_single(size=40, thickness=2, color=(0, 0, 0)):
    img = np.zeros((size, size, 4), dtype=np.uint8); m = size // 2
    cv.line(img, (m, 2), (m, size - 3), (*color, 255), thickness)
    cv.line(img, (m - size // 4, m), (m + size // 4, m), (*color, 255), thickness)
    return img

def draw_single_x(size=40, thickness=2, color=(0, 0, 0)):
    img = np.zeros((size, size, 4), dtype=np.uint8); pad = size // 5
    cv.line(img, (pad, pad), (size - pad, size - pad), (*color, 255), thickness)
    cv.line(img, (size - pad, pad), (pad, size - pad), (*color, 255), thickness)
    return img

def _draw_tall(size, n_slashes, thickness, color):
    """T-stem with N evenly-spaced slashes (used for double/treble/dtr)."""
    w, h = int(size * 0.35), size
    img = np.zeros((h, w, 4), dtype=np.uint8); mx = w // 2
    cv.line(img, (mx, 2), (mx, h - 3), (*color, 255), thickness)              # vertical stem
    cv.line(img, (mx - w // 4, 2), (mx + w // 4, 2), (*color, 255), thickness) # T-bar at top
    if n_slashes:
        offsets = np.linspace(-h // 6, h // 6, n_slashes).astype(int) if n_slashes > 1 else [0]
        for off in offsets:
            cy = h // 2 + off
            cv.line(img, (mx - w // 3, cy + h // 8),
                    (mx + w // 3, cy - h // 8), (*color, 255), thickness)
    return img

def draw_double(size=80, thickness=2, color=(0, 0, 0)):       return _draw_tall(size, 1, thickness, color)
def draw_treble(size=100, thickness=2, color=(0, 0, 0)):      return _draw_tall(size, 2, thickness, color)
def draw_double_treble(size=120, thickness=2, color=(0, 0, 0)): return _draw_tall(size, 3, thickness, color)

def draw_half_double(size=60, thickness=2, color=(0, 0, 0)):
    w, h = int(size * 0.4), size
    img = np.zeros((h, w, 4), dtype=np.uint8); mx = w // 2
    cv.line(img, (mx, 2), (mx, h - 3), (*color, 255), thickness)
    y_bar = h // 3
    cv.line(img, (mx - w // 3, y_bar), (mx + w // 3, y_bar), (*color, 255), thickness)
    cv.line(img, (mx - w // 4, 2), (mx + w // 4, 2), (*color, 255), thickness)
    return img

def draw_fan(n_spokes=5, spoke_len=70, thickness=2, color=(0, 0, 0)):
    spread = math.radians(90)
    w, h = int(spoke_len * 1.6), int(spoke_len * 1.2)
    img = np.zeros((h, w, 4), dtype=np.uint8); base = (w // 2, h - 4)
    for i in range(n_spokes):
        angle = math.pi / 2 + spread / 2 - (spread * i / max(n_spokes - 1, 1))
        ex = int(base[0] + spoke_len * math.cos(angle))
        ey = int(base[1] - spoke_len * math.sin(angle))
        cv.line(img, base, (ex, ey), (*color, 255), thickness)
    return img

def draw_ensemble_chain(n_chains=5, chain_size=20, thickness=2, color=(0, 0, 0)):
    ch_h = int(chain_size * 0.6); gap = 2
    total_h = n_chains * (ch_h + gap) + 4
    w = chain_size + 8
    img = np.zeros((total_h, w, 4), dtype=np.uint8); cx = w // 2
    for i in range(n_chains):
        cy = 2 + ch_h // 2 + i * (ch_h + gap)
        cv.ellipse(img, (cx, cy), (chain_size // 2, ch_h // 2), 0, 0, 360,
                   (*color, 255), thickness)
    return img

def draw_noise_circle(size=30, thickness=2, color=(0, 0, 0)):
    img = np.zeros((size, size, 4), dtype=np.uint8)
    cv.circle(img, (size // 2, size // 2), size // 3, (*color, 255), thickness)
    return img

def draw_noise_number(num=1, size=30, color=(0, 0, 0)):
    img = np.zeros((size, size, 4), dtype=np.uint8); r = size // 2 - 2
    cv.circle(img, (size // 2, size // 2), r, (*color, 255), 1)
    (tw, th), _ = cv.getTextSize(str(num), cv.FONT_HERSHEY_SIMPLEX, size / 60, 1)
    cv.putText(img, str(num), (size // 2 - tw // 2, size // 2 + th // 2),
               cv.FONT_HERSHEY_SIMPLEX, size / 60, (*color, 255), 1, cv.LINE_AA)
    return img

def draw_noise_arrow(size=40, thickness=2, color=(0, 0, 0), direction="right"):
    w, h = size, int(size * 0.4)
    img = np.zeros((h, w, 4), dtype=np.uint8); y = h // 2
    if direction == "right":
        cv.line(img, (2, y), (w - 6, y), (*color, 255), thickness)
        cv.line(img, (w - 10, y - 5), (w - 4, y), (*color, 255), thickness)
        cv.line(img, (w - 10, y + 5), (w - 4, y), (*color, 255), thickness)
    else:
        cv.line(img, (6, y), (w - 2, y), (*color, 255), thickness)
        cv.line(img, (10, y - 5), (4, y), (*color, 255), thickness)
        cv.line(img, (10, y + 5), (4, y), (*color, 255), thickness)
    return img

PROCEDURAL_DRAWERS = {
    "chain":         [draw_chain],
    "double":        [draw_double],
    "double treble": [draw_double_treble],
    "single":        [draw_single, draw_single_x],
    "half_double":   [draw_half_double],
    "treble":        [draw_treble],
    "fan":           [draw_fan],
    "enseble_chain": [draw_ensemble_chain],
    "noise":         [draw_noise_circle, draw_noise_number, draw_noise_arrow],
}

### 1.2  Helpers — symbol retrieval, paste-with-rotation, label conversion

`paste_symbol` is the heart of the generator: it rotates a BGRA glyph around its center, alpha-blends it onto the canvas, and returns the **four rotated corner points** that become the YOLO-OBB label. Because we compute the corners by transforming the original `(0,0)..(w,h)` rectangle through the same rotation matrix used for the warp, the labels are always pixel-perfect — there is no need for a separate post-hoc bounding-box estimation.

In [ ]:
# Templates are loaded lazily — fall back to procedural drawers if absent.
TEMPLATES = {cls: [] for cls in CLASS_MAP}
for cls_name in CLASS_MAP:
    cls_dir = os.path.join(TEMPLATE_DIR, cls_name)
    if os.path.isdir(cls_dir):
        for p in sorted(glob.glob(os.path.join(cls_dir, "*.png"))):
            img = cv.imread(p, cv.IMREAD_UNCHANGED)
            if img is None: continue
            if img.ndim == 2:        img = cv.cvtColor(img, cv.COLOR_GRAY2BGRA)
            elif img.shape[2] == 3:  img = cv.cvtColor(img, cv.COLOR_BGR2BGRA)
            TEMPLATES[cls_name].append(img)


def get_symbol(cls_name, target_h, color=(0, 0, 0), thickness=2):
    """Return a BGRA glyph of approximate height `target_h`.

    50/50 mix between a real PNG template (if any exist for this class) and
    a procedural drawer — keeps the dataset varied even when templates are
    missing for some classes.
    """
    use_template = random.random() < 0.5 and len(TEMPLATES.get(cls_name, [])) > 0
    if use_template:
        tmpl = random.choice(TEMPLATES[cls_name]).copy()
    else:
        drawers = PROCEDURAL_DRAWERS.get(cls_name)
        if not drawers:
            tmpl = np.zeros((20, 20, 4), dtype=np.uint8)
            cv.circle(tmpl, (10, 10), 5, (*color, 255), -1)
        else:
            drawer = random.choice(drawers)
            kwargs = {"color": color, "thickness": thickness}
            # Sensible per-class size ranges so the synthetic charts feel realistic.
            size_ranges = {
                "chain":         (30, 50),  "double":      (60, 100),
                "single":        (25, 45),  "half_double": (45, 70),
                "treble":        (80, 120), "double treble": (100, 140),
            }
            if cls_name in size_ranges:
                kwargs["size"] = random.randint(*size_ranges[cls_name])
            elif cls_name == "fan":
                kwargs = {"n_spokes": random.randint(3, 7),
                          "spoke_len": random.randint(50, 80),
                          "thickness": thickness, "color": color}
            elif cls_name == "enseble_chain":
                kwargs = {"n_chains": random.randint(3, 8),
                          "chain_size": random.randint(15, 25),
                          "thickness": thickness, "color": color}
            elif cls_name == "noise":
                if drawer is draw_noise_number:
                    kwargs = {"num": random.randint(1, 30),
                              "size": random.randint(20, 35), "color": color}
                elif drawer is draw_noise_arrow:
                    kwargs = {"size": random.randint(30, 50), "thickness": thickness,
                              "color": color, "direction": random.choice(["left", "right"])}
                else:
                    kwargs = {"size": random.randint(15, 30),
                              "thickness": thickness, "color": color}
            tmpl = drawer(**kwargs)

    if tmpl.shape[0] < 1 or tmpl.shape[1] < 1:
        return np.zeros((target_h, max(target_h // 2, 10), 4), dtype=np.uint8)
    scale = target_h / tmpl.shape[0]
    return cv.resize(tmpl, (max(int(tmpl.shape[1] * scale), 1), target_h),
                     interpolation=cv.INTER_AREA)


def paste_symbol(canvas, symbol, cx, cy, angle_deg=0):
    """Rotate `symbol` by `angle_deg` and alpha-blend it onto `canvas` at (cx, cy).
    Returns the 4 rotated corner points (pixel coords) — those become the OBB label."""
    h, w = symbol.shape[:2]
    M = cv.getRotationMatrix2D((w / 2, h / 2), -angle_deg, 1.0)
    cos_a, sin_a = abs(M[0, 0]), abs(M[0, 1])
    new_w, new_h = int(h * sin_a + w * cos_a), int(h * cos_a + w * sin_a)
    M[0, 2] += (new_w - w) / 2
    M[1, 2] += (new_h - h) / 2
    rotated = cv.warpAffine(symbol, M, (new_w, new_h),
                            flags=cv.INTER_LINEAR, borderValue=(0, 0, 0, 0))

    x1, y1 = int(cx - new_w / 2), int(cy - new_h / 2)
    ch, cw = canvas.shape[:2]
    sx, sy = max(0, -x1), max(0, -y1)
    ex, ey = min(new_w, cw - x1), min(new_h, ch - y1)
    if sx >= ex or sy >= ey:
        return None  # fully outside canvas

    roi   = canvas[y1 + sy:y1 + ey, x1 + sx:x1 + ex]
    patch = rotated[sy:ey, sx:ex]
    alpha = patch[:, :, 3:4].astype(np.float32) / 255.0
    for c in range(3):
        roi[:, :, c] = (alpha[:, :, 0] * patch[:, :, c] +
                        (1 - alpha[:, :, 0]) * roi[:, :, c]).astype(np.uint8)

    # Project the original (0,0)-(w,h) rectangle through the same M.
    corners = np.array([[0, 0], [w, 0], [w, h], [0, h]], dtype=np.float32)
    pts = np.hstack([corners, np.ones((4, 1), dtype=np.float32)])
    transformed = (M @ pts.T).T
    transformed[:, 0] += x1
    transformed[:, 1] += y1
    return transformed


def obb_to_yolo(corners, img_w, img_h):
    """YOLO-OBB label = 'cls x1 y1 x2 y2 x3 y3 x4 y4', normalised to [0,1]."""
    out = []
    for px, py in corners:
        out.extend([float(np.clip(px / img_w, 0, 1)),
                    float(np.clip(py / img_h, 0, 1))])
    return out


def add_grid_lines(canvas, spacing=30, color=(200, 200, 200), thickness=1):
    h, w = canvas.shape[:2]
    for x in range(0, w, spacing): cv.line(canvas, (x, 0), (x, h), color, thickness)
    for y in range(0, h, spacing): cv.line(canvas, (0, y), (w, y), color, thickness)


def add_aging_texture(canvas, intensity=0.03):
    noise = np.random.normal(0, intensity * 255, canvas.shape).astype(np.float32)
    return np.clip(canvas.astype(np.float32) + noise, 0, 255).astype(np.uint8)

### 1.3  Layout generators

Each generator returns `(canvas_bgr, labels)` where `labels` is a list of `(cls_id, x1,y1, ..., x4,y4)` in **normalised** coords. The seven generators below reproduce the most common chart structures we observed in real photos. They are picked at random with the weights at the bottom of the cell — V-pattern grids and triangular/expanding charts dominate because that is what real charts look like.

In [ ]:
def _add_edge_noise(canvas, labels, img_size, fg, margin):
    """Sprinkle row numbers / arrows / circles along the edges (the 'noise' class)."""
    for i in range(random.randint(0, 5)):
        ny = random.randint(margin, img_size - margin)
        side = random.choice(["left", "right"])
        nx = (random.randint(2, margin - 5) if side == "left"
              else random.randint(img_size - margin + 5, img_size - 5))
        kind = random.choice(["number", "arrow", "circle"])
        if kind == "number":
            sym = draw_noise_number(num=i + 1, size=random.randint(18, 28), color=fg)
        elif kind == "arrow":
            sym = draw_noise_arrow(size=random.randint(25, 40), color=fg,
                                   direction="right" if side == "left" else "left")
        else:
            sym = draw_noise_circle(size=random.randint(12, 22), color=fg)
        c = paste_symbol(canvas, sym, nx, ny, random.gauss(0, 3))
        if c is not None:
            labels.append((CLASS_MAP["noise"], *obb_to_yolo(c, img_size, img_size)))


def generate_dense_rows(img_size=640, palette=None):
    """Standard horizontal rows of stitches — the most common chart layout."""
    palette = palette or random.choice(list(PALETTES.values()))
    bg = np.full((img_size, img_size, 3), palette["bg"], dtype=np.uint8); fg = palette["fg"]
    if random.random() < 0.6:
        add_grid_lines(bg, spacing=random.randint(20, 40),
                       color=tuple(int(c * 0.9) for c in palette["bg"]))
    labels = []
    stitch_h = random.randint(25, 50); row_gap = stitch_h + random.randint(5, 15)
    margin = random.randint(20, 50)
    row_classes = random.choices(
        ["chain", "double", "single", "half_double", "treble"],
        weights=[5, 4, 3, 2, 1], k=random.randint(3, 8))
    y, row_num = margin, 0
    while y + stitch_h < img_size - margin:
        cls = row_classes[row_num % len(row_classes)]
        x = margin + random.randint(-5, 5)
        stitch_w = int(stitch_h * random.uniform(0.3, 0.8))
        gap_x = stitch_w + random.randint(2, 8)
        while x + stitch_w < img_size - margin:
            sym = get_symbol(cls, stitch_h, color=fg)
            c = paste_symbol(bg, sym, x + stitch_w // 2, y + stitch_h // 2,
                             random.gauss(0, 3))
            if c is not None:
                labels.append((CLASS_MAP[cls], *obb_to_yolo(c, img_size, img_size)))
            x += gap_x
        y += row_gap; row_num += 1
    _add_edge_noise(bg, labels, img_size, fg, margin)
    return add_aging_texture(bg), labels


def generate_v_pattern_grid(img_size=640, palette=None):
    """Pairs of rotated doubles forming V-stitches — the dominant real-world pattern."""
    palette = palette or random.choice(list(PALETTES.values()))
    bg = np.full((img_size, img_size, 3), palette["bg"], dtype=np.uint8); fg = palette["fg"]
    labels = []
    stitch_h = random.randint(35, 55); v_spread = random.randint(15, 30)
    row_gap = stitch_h + random.randint(8, 20); col_gap = random.randint(25, 45)
    margin = random.randint(25, 50)
    y, row_num = margin + stitch_h // 2, 0
    while y + stitch_h // 2 < img_size - margin:
        x = margin + ((col_gap // 2) if row_num % 2 == 1 else 0)
        while x + col_gap < img_size - margin:
            for ang, dx in [(v_spread, -col_gap // 6), (-v_spread, col_gap // 6)]:
                sym = get_symbol("double", stitch_h, color=fg)
                c = paste_symbol(bg, sym, x + dx, y, ang + random.gauss(0, 3))
                if c is not None:
                    labels.append((CLASS_MAP["double"], *obb_to_yolo(c, img_size, img_size)))
            if random.random() < 0.7:
                ch = get_symbol("chain", int(stitch_h * 0.35), color=fg)
                c = paste_symbol(bg, ch, x, y + stitch_h // 3, random.gauss(0, 5))
                if c is not None:
                    labels.append((CLASS_MAP["chain"], *obb_to_yolo(c, img_size, img_size)))
            x += col_gap
        y += row_gap; row_num += 1
    _add_edge_noise(bg, labels, img_size, fg, margin)
    return add_aging_texture(bg), labels


def generate_triangular_chart(img_size=640, palette=None):
    """Rows expand upward — mimics shawls/blankets that grow from a centre point."""
    palette = palette or random.choice(list(PALETTES.values()))
    bg = np.full((img_size, img_size, 3), palette["bg"], dtype=np.uint8); fg = palette["fg"]
    labels = []
    stitch_h = random.randint(28, 45); row_gap = stitch_h + random.randint(5, 12)
    margin = random.randint(20, 40); n_rows = random.randint(6, 14)
    min_n, max_n = random.randint(2, 5), random.randint(15, 30)
    v_spread = random.randint(12, 25)
    for row_i in range(n_rows):
        t = row_i / max(n_rows - 1, 1)
        n = int(min_n + t * (max_n - min_n))
        y = img_size - margin - row_i * row_gap
        if y - stitch_h // 2 < margin: break
        row_w = n * (stitch_h * 0.6); x_start = (img_size - row_w) / 2
        gap = row_w / max(n, 1)
        for j in range(n):
            x = x_start + j * gap + gap / 2
            if random.random() < 0.7:
                for sgn, dx in [(1, -4), (-1, 4)]:
                    sym = get_symbol("double", stitch_h, color=fg)
                    c = paste_symbol(bg, sym, int(x + dx), y, sgn * v_spread + random.gauss(0, 2))
                    if c is not None:
                        labels.append((CLASS_MAP["double"], *obb_to_yolo(c, img_size, img_size)))
            else:
                cls = random.choices(["double", "single", "chain", "half_double", "treble"],
                                     weights=[4, 2, 3, 2, 1])[0]
                sym = get_symbol(cls, stitch_h, color=fg)
                c = paste_symbol(bg, sym, int(x), y, random.gauss(0, 5))
                if c is not None:
                    labels.append((CLASS_MAP[cls], *obb_to_yolo(c, img_size, img_size)))
    if random.random() < 0.4:
        for side_x in [margin // 2, img_size - margin // 2]:
            ec = get_symbol("enseble_chain", random.randint(60, 120), color=fg)
            c = paste_symbol(bg, ec, side_x, img_size // 2, random.gauss(0, 5))
            if c is not None:
                labels.append((CLASS_MAP["enseble_chain"], *obb_to_yolo(c, img_size, img_size)))
    _add_edge_noise(bg, labels, img_size, fg, margin)
    return add_aging_texture(bg), labels


# Three more generators omitted for brevity here — see YOLO_OBB_synthetic_gen.ipynb
# for generate_fan_grid / generate_mixed_chart / generate_chain_grid / generate_single_fan.

GENERATORS = [
    (generate_v_pattern_grid,   0.40),  # V-stitch (dominant in real charts)
    (generate_triangular_chart, 0.30),  # expanding triangular layouts
    (generate_dense_rows,       0.30),  # standard dense rows
]
gen_funcs, gen_weights = zip(*GENERATORS)

### 1.4  Augmentation, generation loop, `data.yaml`

Augmentations are deliberately *photographic* (brightness, contrast, blur, JPEG, small global rotation, random crop-and-resize). They simulate phone-camera variability without distorting the geometry too much, so the rotated-corner labels stay valid.

The final cell of Step 1 also writes `data.yaml`, which is the **single point of contact** between the dataset and the trainer.

In [ ]:
def augment_image(img):
    out = img.copy()
    if random.random() < 0.5:
        out = np.clip(random.uniform(0.85, 1.15) * out.astype(np.float32)
                      + random.randint(-15, 15), 0, 255).astype(np.uint8)
    if random.random() < 0.3:
        out = cv.GaussianBlur(out, (random.choice([3, 5]),) * 2, 0)
    if random.random() < 0.3:
        _, enc = cv.imencode('.jpg', out, [cv.IMWRITE_JPEG_QUALITY, random.randint(50, 90)])
        out = cv.imdecode(enc, cv.IMREAD_COLOR)
    if random.random() < 0.2:
        h, w = out.shape[:2]
        M = cv.getRotationMatrix2D((w / 2, h / 2), random.uniform(-3, 3), 1.0)
        out = cv.warpAffine(out, M, (w, h), borderValue=(255, 255, 255))
    return out


def generate_dataset(n_images, img_dir, lbl_dir, prefix="syn"):
    for i in range(n_images):
        gen_fn = random.choices(gen_funcs, weights=gen_weights, k=1)[0]
        try:
            canvas, labels = gen_fn(img_size=IMG_SIZE,
                                    palette=random.choice(list(PALETTES.values())))
        except Exception as e:
            print(f"  [warn] {gen_fn.__name__}: {e}"); continue
        canvas = augment_image(canvas)
        valid = [lbl for lbl in labels
                 if (max(lbl[1::2]) - min(lbl[1::2])) > 0.005
                 and (max(lbl[1:9:2]) - min(lbl[1:9:2])) > 0.005]
        if not valid: continue
        fname = f"{prefix}_{i:04d}"
        cv.imwrite(os.path.join(img_dir, f"{fname}.png"), canvas)
        with open(os.path.join(lbl_dir, f"{fname}.txt"), "w") as f:
            for lbl in valid:
                f.write(f"{int(lbl[0])} " + " ".join(f"{c:.6f}" for c in lbl[1:]) + "\n")
        if (i + 1) % 50 == 0:
            print(f"  generated {i + 1}/{n_images}")


# Execute the generator. Skip if the dataset is already produced and only
# the data.yaml refresh is wanted.
RUN_GENERATION = True
if RUN_GENERATION:
    print(f"Generating {NUM_TRAIN} train + {NUM_VAL} val images …")
    generate_dataset(NUM_TRAIN, TRAIN_IMG, TRAIN_LBL, prefix="syn_train")
    generate_dataset(NUM_VAL,   VAL_IMG,   VAL_LBL,   prefix="syn_val")

# Write data.yaml — the only file YOLO needs to find the dataset.
data_yaml = {
    "path":  OUTPUT_DIR,
    "train": "train/images",
    "val":   "val/images",
    "names": ID_TO_NAME,
}
yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)
print("Wrote", yaml_path)

### 1.5  (Optional) mix in real labelled images

Whenever a hand-labelled batch comes back from Label Studio (Step 3), copy it into the same `train/` and `val/` folders. The model then sees both: synthetic for variety and balance, real for fidelity to the actual problem.

In [ ]:
# ── Optional: copy real labelled data into the training set ───────────────
REAL_DATA_DIR = os.path.join(PROJECT_DIR, "..", "project-11-at-2026-04-10-14-28-21a55b58")
if os.path.isdir(REAL_DATA_DIR):
    real_imgs = sorted(glob.glob(os.path.join(REAL_DATA_DIR, "images", "*.png")) +
                       glob.glob(os.path.join(REAL_DATA_DIR, "images", "*.jpg")))
    n_val = max(1, len(real_imgs) // 4)
    val_idx = set(random.sample(range(len(real_imgs)), n_val))
    for idx, img_path in enumerate(real_imgs):
        stem = Path(img_path).stem
        lbl  = os.path.join(REAL_DATA_DIR, "labels", stem + ".txt")
        if not os.path.isfile(lbl): continue
        dst_dir_img, dst_dir_lbl = (VAL_IMG, VAL_LBL) if idx in val_idx else (TRAIN_IMG, TRAIN_LBL)
        shutil.copy2(img_path, os.path.join(dst_dir_img, f"real_{stem}.png"))
        shutil.copy2(lbl,      os.path.join(dst_dir_lbl, f"real_{stem}.txt"))
    print(f"Mixed {len(real_imgs)} real images into the dataset.")
else:
    print("No real-data dir present yet — synthetic-only training.")

In [ ]:
# ── Sanity check: count files + class distribution ────────────────────────
class_counts = Counter()
for lbl in glob.glob(os.path.join(TRAIN_LBL, "*.txt")) + glob.glob(os.path.join(VAL_LBL, "*.txt")):
    with open(lbl) as f:
        for line in f:
            parts = line.split()
            if len(parts) >= 9:
                class_counts[int(parts[0])] += 1

print(f"Train: {len(glob.glob(os.path.join(TRAIN_IMG, '*.png')))} imgs, "
      f"Val: {len(glob.glob(os.path.join(VAL_IMG, '*.png')))} imgs\n")
print("Class distribution:")
for cls_id in sorted(class_counts):
    print(f"  {cls_id} {ID_TO_NAME[cls_id]:<14}  {class_counts[cls_id]}")

---
# Step 2 — Training on real + generated data

The model is **YOLOv8n-OBB** — Ultralytics' nano variant of YOLOv8 with the OBB head. We use the OBB head because crochet stitches are frequently rotated (V-stitches, fans, and slanted doubles), and rotation-aware boxes give cleaner predictions than axis-aligned boxes plus a separate angle head.

Two equivalent training entry points are documented below: a **local MPS** path that runs end-to-end on an Apple-Silicon laptop and a **Colab GPU** path that finishes the same job in ~15 minutes by saving weights straight to Google Drive. Pick whichever is convenient — both produce a `runs/obb/<name>/weights/best.pt` that Step 3 will load.

**Critical hyperparameters**

* `imgsz=640` — matches the synthetic-image side. Bigger inputs cost VRAM with little accuracy gain on these glyphs.
* `degrees=180` — the augmentation that lets the model see every possible glyph orientation. Without this, V-stitches (rotated doubles) are systematically mis-detected.
* `batch=4` (laptop) / `batch=16` (GPU). MPS will hang at higher batches on 16 GB machines.
* `epochs=10` for a quick local sanity-check; `epochs=150` on the GPU for the production run.

### 2.1  Local training (MPS / Apple-Silicon)

In [ ]:
# ── Local training on Apple Silicon (MPS) ─────────────────────────────────
# Expect ~5 hours for 10 epochs on an M1 Pro. Reduce `epochs` for a smoke test.
import torch
from ultralytics import YOLO

if torch.backends.mps.is_available():
    torch.mps.empty_cache()

model = YOLO("yolov8n-obb.pt")           # Ultralytics auto-downloads the COCO-OBB pretrain
model.train(
    data    = yaml_path,                  # path written at the end of Step 1
    epochs  = 10,
    batch   = 4,                          # M1 Pro: keep it small; 16 GB RAM hangs at >4
    imgsz   = 640,
    device  = "mps",
    degrees = 180,                        # crucial for rotated stitches
    workers = 4,
    plots   = True,
)

### 2.2  Colab GPU training (recommended for the full 150-epoch run)

The cell below mounts Google Drive, points the trainer at a `data.yaml` that lives on Drive, and writes weights back to Drive so they survive a session reset.

In [ ]:
# ── Colab GPU training — run this cell only inside Google Colab ──────────
# from google.colab import drive
# drive.mount('/content/drive')
#
# DRIVE_BASE = '/content/drive/MyDrive/Crochet_data'
# yaml_path  = f'{DRIVE_BASE}/project_yolo_obb/data.yaml'
# save_dir   = f'{DRIVE_BASE}/runs'
#
# from ultralytics import YOLO
# model = YOLO('yolov8n-obb.pt')
# model.train(
#     data    = yaml_path,
#     epochs  = 150,
#     imgsz   = 640,
#     batch   = 16,
#     device  = 0,           # CUDA device 0
#     degrees = 180,
#     workers = 8,
#     cache   = True,        # caches resized images in RAM — big speedup
#     project = save_dir,    # weights land on Drive, survive disconnects
#     name    = 'obb_train',
# )
# best_model_path = str(model.trainer.best)
# print('Best weights ->', best_model_path)

### 2.3  Validation — confusion matrix and per-class metrics

After training, walk the val split and build a `(n_cls + 1) × (n_cls + 1)` confusion matrix where the extra row/column is "background / missed". Greedy IoU matching (highest-confidence prediction first, IoU ≥ 0.50) yields per-class TP / FP / FN counts, from which Precision / Recall / F1 fall out cleanly. We also visualise the matrix and the per-class P/R/F1 bars — these were the plots in the Colab notebook.

In [ ]:
def evaluate_obb(weights_path, yaml_path, conf=0.25, iou_thresh=0.50):
    """Greedy IoU matching against YOLO-OBB ground truth → per-class metrics."""
    from ultralytics import YOLO

    model = YOLO(weights_path)
    with open(yaml_path) as f:
        cfg = yaml.safe_load(f)
    root = cfg.get("path", os.path.dirname(yaml_path))
    val_imgs_dir = cfg["val"] if os.path.isabs(cfg["val"]) else os.path.join(root, cfg["val"])
    val_lbls_dir = val_imgs_dir.replace(os.sep + "images", os.sep + "labels")

    id2name = {int(k): v for k, v in model.names.items()}
    n_cls   = len(id2name)
    BG      = n_cls
    cm      = np.zeros((n_cls + 1, n_cls + 1), dtype=int)

    def _bbox_iou(a, b):
        ax1, ay1 = a.min(0); ax2, ay2 = a.max(0)
        bx1, by1 = b.min(0); bx2, by2 = b.max(0)
        ix1, iy1 = max(ax1, bx1), max(ay1, by1)
        ix2, iy2 = min(ax2, bx2), min(ay2, by2)
        if ix2 <= ix1 or iy2 <= iy1: return 0.0
        inter = (ix2 - ix1) * (iy2 - iy1)
        union = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
        return inter / union if union > 0 else 0.0

    img_files = sorted(glob.glob(os.path.join(val_imgs_dir, "*.png")) +
                       glob.glob(os.path.join(val_imgs_dir, "*.jpg")))
    for img_path in img_files:
        img = cv.imread(img_path);   ih, iw = img.shape[:2]
        stem = Path(img_path).stem
        gt_list = []
        lbl = os.path.join(val_lbls_dir, stem + ".txt")
        if os.path.exists(lbl):
            with open(lbl) as f:
                for line in f:
                    parts = line.split()
                    if len(parts) < 9: continue
                    corn = np.array(parts[1:9], float).reshape(4, 2)
                    corn[:, 0] *= iw; corn[:, 1] *= ih
                    gt_list.append((int(parts[0]), corn))

        res = model.predict(img, conf=conf, verbose=False)[0]
        preds = [(int(b.cls[0]),
                  b.xyxyxyxy.cpu().numpy().reshape(4, 2).astype(float),
                  float(b.conf[0])) for b in res.obb]

        matched_gt = set(); matched_pred = set()
        for pi, (pcls, pcorn, _) in sorted(enumerate(preds), key=lambda x: -x[1][2]):
            best_iou, best_gi = 0.0, -1
            for gi, (_, gcorn) in enumerate(gt_list):
                if gi in matched_gt: continue
                iou = _bbox_iou(pcorn, gcorn)
                if iou > best_iou: best_iou, best_gi = iou, gi
            if best_iou >= iou_thresh:
                cm[gt_list[best_gi][0], pcls] += 1
                matched_gt.add(best_gi); matched_pred.add(pi)
            else:
                cm[BG, pcls] += 1
        for gi, (gcls, _) in enumerate(gt_list):
            if gi not in matched_gt: cm[gcls, BG] += 1

    tp = np.array([cm[i, i] for i in range(n_cls)], dtype=float)
    fp = np.array([cm[:, i].sum() - cm[i, i] for i in range(n_cls)], dtype=float)
    fn = np.array([cm[i, :].sum() - cm[i, i] for i in range(n_cls)], dtype=float)
    prec = np.where(tp + fp > 0, tp / (tp + fp), 0.0)
    rec  = np.where(tp + fn > 0, tp / (tp + fn), 0.0)
    f1   = np.where(prec + rec > 0, 2 * prec * rec / (prec + rec), 0.0)

    labels = [id2name[i] for i in range(n_cls)]
    print(f"{'Class':<16}{'TP':>6}{'FP':>6}{'FN':>6}{'P':>10}{'R':>8}{'F1':>8}")
    for i, c in enumerate(labels):
        print(f"{c:<16}{int(tp[i]):>6}{int(fp[i]):>6}{int(fn[i]):>6}"
              f"{prec[i]:>10.3f}{rec[i]:>8.3f}{f1[i]:>8.3f}")
    print(f"{'macro avg':<16}{int(tp.sum()):>6}{int(fp.sum()):>6}{int(fn.sum()):>6}"
          f"{prec.mean():>10.3f}{rec.mean():>8.3f}{f1.mean():>8.3f}")

    # Heatmap
    tick = labels + ["bg"]
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm, cmap="Blues")
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.set_xticks(range(n_cls + 1)); ax.set_yticks(range(n_cls + 1))
    ax.set_xticklabels(tick, rotation=45, ha='right'); ax.set_yticklabels(tick)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth")
    ax.set_title(f"OBB confusion matrix (macro F1 = {f1.mean():.3f})")
    for r in range(n_cls + 1):
        for c in range(n_cls + 1):
            if cm[r, c]:
                ax.text(c, r, cm[r, c], ha='center', va='center', fontsize=7,
                        color="white" if cm[r, c] > cm.max() / 2 else "black")
    plt.tight_layout(); plt.show()
    return cm, prec, rec, f1


# Uncomment after a successful training run:
# WEIGHTS = os.path.join(PROJECT_DIR, "runs", "obb", "obb_train23", "weights", "best.pt")
# evaluate_obb(WEIGHTS, yaml_path, conf=DEFAULT_CONF, iou_thresh=DEFAULT_IOU)

---
# Step 3 — Real image example: adaptive tiled inference

Real photos rarely come in convenient 640×640 squares. A typical chart photo is 460×1000+ px, which means each stitch occupies only ~20 px when the image is downsampled to YOLO's input size — far smaller than the 80–120 px stitches the model was trained on. The fix is **adaptive tiling**:

1. **Estimate** the median stitch size with a quick low-confidence pass on the downsampled image.
2. Compute the **effective tile size** so that stitches will appear at the *training* scale (~100 px) once each tile is upscaled to 640 inside YOLO.
3. **Slice** the image into overlapping tiles of that effective size, infer each, **translate** detections back into original-image coordinates, and **NMS** across tiles.

The same algorithm lives in the `crochet.detection` module, so the Streamlit app and the MCP server reuse it verbatim. The cells below reproduce the algorithm inline for clarity.

In [ ]:
# ── Detection dataclass ───────────────────────────────────────────────────
@dataclass
class Detection:
    corners: np.ndarray   # (4, 2) — 4 rotated corners in *original-image* coords
    cls_id: int
    cls_name: str
    confidence: float

In [ ]:
# ── Adaptive tiling primitives ────────────────────────────────────────────

def _rect_iou(a, b):
    """AABB IoU on two (4,2) corner arrays — fast surrogate for full OBB IoU."""
    ax1, ay1 = a.min(0); ax2, ay2 = a.max(0)
    bx1, by1 = b.min(0); bx2, by2 = b.max(0)
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    if ix2 <= ix1 or iy2 <= iy1: return 0.0
    inter = (ix2 - ix1) * (iy2 - iy1)
    union = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / union if union > 0 else 0.0


def _nms(detections, iou_threshold=0.5):
    """Greedy class-aware NMS."""
    detections = sorted(detections, key=lambda d: d.confidence, reverse=True)
    kept = []
    while detections:
        best = detections.pop(0)
        kept.append(best)
        detections = [d for d in detections
                      if d.cls_id != best.cls_id
                      or _rect_iou(best.corners, d.corners) < iou_threshold]
    return kept


def _tile_starts(length, tile_size, stride):
    starts = list(range(0, length - tile_size, stride))
    if not starts or starts[-1] + tile_size < length:
        starts.append(max(0, length - tile_size))
    return starts


def _infer_tile(model, tile, x_off, y_off, conf):
    res = model.predict(tile, conf=conf, verbose=False)[0]
    out = []
    for box in res.obb:
        corners = box.xyxyxyxy.cpu().numpy().reshape(4, 2).astype(np.float32)
        corners[:, 0] += x_off
        corners[:, 1] += y_off
        cls_id = int(box.cls[0])
        out.append(Detection(corners, cls_id, res.names[cls_id], float(box.conf[0])))
    return out


def estimate_stitch_size(model, image, tile_size=640, conf=0.15):
    """Median short-axis length of OBBs detected on a downsampled image."""
    h, w = image.shape[:2]
    scale = tile_size / max(h, w)
    small = cv.resize(image, (int(w * scale), int(h * scale))) if scale < 1.0 else image
    if scale >= 1.0: scale = 1.0
    res = model.predict(small, conf=conf, verbose=False)[0]
    if len(res.obb) == 0: return None
    sizes = []
    for box in res.obb:
        c = box.xyxyxyxy.cpu().numpy().reshape(4, 2)
        sizes.append(min(np.linalg.norm(c[0] - c[1]), np.linalg.norm(c[1] - c[2])))
    return float(np.median(sizes)) / scale


def predict_tiled(model, image, tile_size=640, overlap=0.2,
                  conf=0.25, iou_threshold=0.5):
    h, w = image.shape[:2]
    if h <= tile_size and w <= tile_size:
        return _infer_tile(model, image, 0, 0, conf)
    stride = max(1, int(tile_size * (1 - overlap)))
    out = []
    for y1 in _tile_starts(h, tile_size, stride):
        for x1 in _tile_starts(w, tile_size, stride):
            tile = image[y1:min(y1 + tile_size, h), x1:min(x1 + tile_size, w)]
            out.extend(_infer_tile(model, tile, x1, y1, conf))
    return _nms(out, iou_threshold)


def predict_adaptive(model, image, target_stitch_px=DEFAULT_TARGET_STITCH_PX,
                     tile_size=DEFAULT_TILE_SIZE, overlap=DEFAULT_OVERLAP,
                     conf=DEFAULT_CONF, iou_threshold=DEFAULT_IOU):
    """Estimate stitch size, then choose a tile size so stitches appear at the
    training scale inside each YOLO call."""
    h, w = image.shape[:2]
    est = estimate_stitch_size(model, image, tile_size=tile_size, conf=min(conf, 0.15))
    if est is None or est <= 0:
        print("  [adaptive] size-estimation found nothing — falling back to fixed tiling.")
        return predict_tiled(model, image, tile_size, overlap, conf, iou_threshold)

    eff = max(64, int(tile_size * est / target_stitch_px))
    print(f"  [adaptive] est_stitch={est:.1f}px  →  effective_tile={eff}px  "
          f"(target {target_stitch_px}px at {tile_size})")
    if eff >= max(h, w):
        return _infer_tile(model, image, 0, 0, conf)
    return predict_tiled(model, image, tile_size=eff, overlap=overlap,
                         conf=conf, iou_threshold=iou_threshold)

### 3.1  SVG-style symbol drawer for the reconstructed scheme

Once the boxes are in original-image coordinates, we re-render the chart as a clean SVG-like figure: each detection becomes a procedurally-drawn glyph at the correct centre, size, and rotation. This is the deliverable users care about — not the raw bounding boxes, but a publication-quality stitch chart.

In [ ]:
def draw_svg_icon(ax, label, x, y, w, h, angle_deg, color):
    t = transforms.Affine2D().rotate_deg_around(x, y, angle_deg) + ax.transData
    if label == "chain":
        ax.add_patch(patches.Ellipse((x, y), w * 0.8, h * 0.4,
                                     fill=False, color=color, linewidth=2, transform=t))
    elif label in ("half_double", "double", "double treble", "treble", "fan"):
        ax.plot([x, x], [y - h / 2, y + h / 2], color=color, lw=2, transform=t)
        ax.plot([x - w / 3, x + w / 3], [y - h / 2, y - h / 2], color=color, lw=2, transform=t)
        if label == "double":
            ax.plot([x - w / 4, x + w / 4], [y - h / 8, y + h / 8],
                    color=color, lw=1.5, transform=t)
        elif label == "treble":
            ax.plot([x - w / 4, x + w / 4], [y - h / 4, y - h / 12],
                    color=color, lw=1.5, transform=t)
            ax.plot([x - w / 4, x + w / 4], [y + h / 12, y + h / 4],
                    color=color, lw=1.5, transform=t)
        elif label == "double treble":
            for y_off in [(-h / 3, -h / 6), (-h / 12, h / 12), (h / 6, h / 3)]:
                ax.plot([x - w / 4, x + w / 4], [y + y_off[0], y + y_off[1]],
                        color=color, lw=1.5, transform=t)
        elif label == "fan":
            ax.plot([x, x - w / 3], [y + h / 2, y - h / 2], color=color, lw=1.5,
                    transform=t, alpha=0.6)
            ax.plot([x, x + w / 3], [y + h / 2, y - h / 2], color=color, lw=1.5,
                    transform=t, alpha=0.6)
    elif label == "enseble_chain":
        n = max(2, int(h / max(w * 0.5, 1)))
        oh = h / n
        for i in range(n):
            cy = y - h / 2 + oh * (i + 0.5)
            ax.add_patch(patches.Ellipse((x, cy), w * 0.6, oh * 0.7,
                                         fill=False, color=color, linewidth=1.5, transform=t))
    elif label == "noise":
        ax.add_patch(patches.Circle((x, y), min(w, h) * 0.3,
                                    fill=False, color=color, linewidth=1.5, transform=t))
    else:  # single + fallback
        ax.plot([x - w / 3, x + w / 3], [y - h / 3, y + h / 3], color=color, lw=2, transform=t)
        ax.plot([x - w / 3, x + w / 3], [y + h / 3, y - h / 3], color=color, lw=2, transform=t)

### 3.2  Run inference on a real photo

Edit `WEIGHTS` to point at the `best.pt` produced by Step 2 and `TEST_IMG` to any photo you want to inspect. The cell shows the **two-panel** layout used in every demo: detections overlaid on the original image (left), reconstructed clean scheme (right).

In [ ]:
from ultralytics import YOLO

WEIGHTS  = os.path.join(PROJECT_DIR, "runs", "obb", "obb_train23", "weights", "best.pt")
TEST_IMG = os.path.join(PROJECT_DIR, "data", "raw", "easy", "0.png")

model = YOLO(WEIGHTS)
image = cv.imread(TEST_IMG)
detections = predict_adaptive(model, image,
                              target_stitch_px=DEFAULT_TARGET_STITCH_PX,
                              conf=DEFAULT_CONF,
                              iou_threshold=DEFAULT_IOU)
print(f"Detected {len(detections)} stitches on {Path(TEST_IMG).name}")

h, w = image.shape[:2]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

ax1.imshow(cv.cvtColor(image, cv.COLOR_BGR2RGB))
ax1.set_title("YOLO-OBB detections (adaptive tiling)")
ax2.set_facecolor("white"); ax2.set_xlim(0, w); ax2.set_ylim(h, 0)
ax2.set_title("Reconstructed scheme")

for det in detections:
    cfg = CLASS_CONFIG.get(det.cls_name, {"abbr": "??", "color": DEFAULT_COLOR})
    color, abbr = cfg["color"], cfg["abbr"]
    closed = np.vstack([det.corners, det.corners[0]])
    ax1.plot(closed[:, 0], closed[:, 1], color=color, linewidth=2)
    ax1.text(det.corners[0, 0], det.corners[0, 1] - 5, abbr, color="white",
             fontsize=8, fontweight="bold",
             bbox=dict(facecolor=color, edgecolor="none", alpha=0.7))
    centre = det.corners.mean(axis=0)
    width  = np.linalg.norm(det.corners[0] - det.corners[1])
    height = np.linalg.norm(det.corners[0] - det.corners[3])
    angle  = np.degrees(np.arctan2(det.corners[1, 1] - det.corners[0, 1],
                                   det.corners[1, 0] - det.corners[0, 0]))
    draw_svg_icon(ax2, det.cls_name, centre[0], centre[1], width, height, angle, color)

ax1.axis("off"); ax2.axis("off")
plt.tight_layout(); plt.show()

### 3.3  Tile a *large* photo and emit a Label Studio import

For images that are too big to inspect at a single glance, the project ships a "tile + predict + import-into-Label-Studio" loop. It writes each tile and its predictions to `mydata/test_data/big_tiles/` (the folder Label Studio's Docker container has bind-mounted at `/label-studio/data`), and produces a single `tasks.json` file that pre-populates each task with rotated rectangles. Annotators then *correct* the model's predictions instead of drawing every box from scratch — every correction becomes a fresh real-data sample for the next training round.

This closes the loop:

```
YOLO predicts → annotator corrects → new labels feed back into Step 1's training mix → YOLO improves
```

In [ ]:
def obb_corners_to_rotated_rect(corners_norm, img_w, img_h):
    """YOLO 4-corner OBB (normalised) → Label Studio RectangleLabels rotated rect."""
    pts = [(c[0] * img_w, c[1] * img_h) for c in corners_norm]
    anchor_idx, anchor = sorted(enumerate(pts), key=lambda p: (p[1][1], p[1][0]))[0]
    n_idx = (anchor_idx + 1) % 4
    p_idx = (anchor_idx + 3) % 4
    next_pt, prev_pt = pts[n_idx], pts[p_idx]
    ang_next = math.atan2(next_pt[1] - anchor[1], next_pt[0] - anchor[0])
    ang_prev = math.atan2(prev_pt[1] - anchor[1], prev_pt[0] - anchor[0])
    if abs(ang_next) <= abs(ang_prev):
        wv, hv, rot = ((next_pt[0] - anchor[0], next_pt[1] - anchor[1]),
                       (prev_pt[0] - anchor[0], prev_pt[1] - anchor[1]),
                       math.degrees(ang_next))
    else:
        wv, hv, rot = ((prev_pt[0] - anchor[0], prev_pt[1] - anchor[1]),
                       (next_pt[0] - anchor[0], next_pt[1] - anchor[1]),
                       math.degrees(ang_prev))
    if rot < 0: rot += 360
    return {"x":      anchor[0] / img_w * 100,
            "y":      anchor[1] / img_h * 100,
            "width":  math.hypot(*wv) / img_w * 100,
            "height": math.hypot(*hv) / img_h * 100,
            "rotation": rot}


# ── Cookbook: tile every image in data/raw/big/ + emit tasks.json ─────────
INPUT_DIR  = os.path.join(PROJECT_DIR, "data", "raw", "big")
OUT_ROOT   = Path(os.path.join(PROJECT_DIR, "mydata", "test_data", "big_tiles"))
URL_PREFIX = "/data/local-files/?d=test_data/big_tiles/images/"
(OUT_ROOT / "images").mkdir(parents=True, exist_ok=True)
(OUT_ROOT / "labels").mkdir(parents=True, exist_ok=True)
with open(OUT_ROOT / "classes.txt", "w") as f:
    f.write("\n".join(ID_TO_NAME[i] for i in range(NUM_CLASSES)) + "\n")


def tile_and_save(image_path, model, out_root, tile_size=640,
                  target_stitch_px=DEFAULT_TARGET_STITCH_PX, overlap=0.25,
                  conf=0.20, iou_nms=0.45, min_tile=128):
    image = cv.imread(str(image_path))
    if image is None: return 0, 0
    h, w = image.shape[:2]
    stem = Path(image_path).stem.replace(" ", "_")

    est = estimate_stitch_size(model, image, tile_size=tile_size, conf=0.15)
    eff = (min(tile_size, min(h, w)) if not est or est <= 0
           else min(min(h, w), max(min_tile, int(tile_size * est / target_stitch_px))))
    if eff >= max(h, w):
        x_starts, y_starts = [0], [0]
    else:
        stride = max(1, int(eff * (1 - overlap)))
        y_starts = _tile_starts(h, eff, stride)
        x_starts = _tile_starts(w, eff, stride)

    print(f"  [{stem}] {w}×{h}  est={est}  eff_tile={eff}  tiles={len(y_starts)}×{len(x_starts)}")
    n_tiles = n_dets = 0
    for yi, y1 in enumerate(y_starts):
        for xi, x1 in enumerate(x_starts):
            y2, x2 = min(y1 + eff, h), min(x1 + eff, w)
            tile = image[y1:y2, x1:x2]
            th, tw = tile.shape[:2]
            res = model.predict(tile, conf=conf, iou=iou_nms, verbose=False)[0]
            lines = []
            for box in res.obb:
                corn = box.xyxyxyxy.cpu().numpy().reshape(4, 2).astype(np.float32)
                coords = []
                for cx_, cy_ in corn:
                    coords += [float(np.clip(cx_ / tw, 0, 1)),
                               float(np.clip(cy_ / th, 0, 1))]
                lines.append(f"{int(box.cls[0])} " + " ".join(f"{c:.6f}" for c in coords))
            tile_name = f"{stem}_tile_{n_tiles:03d}"
            cv.imwrite(str(out_root / "images" / f"{tile_name}.jpg"), tile,
                       [cv.IMWRITE_JPEG_QUALITY, 92])
            with open(out_root / "labels" / f"{tile_name}.txt", "w") as f:
                f.write("\n".join(lines) + ("\n" if lines else ""))
            n_tiles += 1; n_dets += len(lines)
    return n_tiles, n_dets


def build_ls_tasks(out_root, class_names, url_prefix):
    tasks = []
    for img_path in sorted((out_root / "images").glob("*.jpg")):
        img = cv.imread(str(img_path));  ih, iw = img.shape[:2]
        lbl = out_root / "labels" / (img_path.stem + ".txt")
        results = []
        if lbl.exists():
            for line in lbl.read_text().splitlines():
                parts = line.split()
                if len(parts) < 9: continue
                cls_id = int(parts[0])
                if not (0 <= cls_id < len(class_names)): continue
                coords = list(map(float, parts[1:9]))
                rect = obb_corners_to_rotated_rect(
                    [(coords[2 * i], coords[2 * i + 1]) for i in range(4)], iw, ih)
                results.append({
                    "original_width": iw, "original_height": ih, "image_rotation": 0,
                    "value": {**rect, "rectanglelabels": [class_names[cls_id]]},
                    "from_name": "label", "to_name": "image", "type": "rectanglelabels",
                })
        task = {"data": {"image": url_prefix + img_path.name}}
        if results:
            task["predictions"] = [{"model_version": "yolo-obb-auto", "result": results}]
        tasks.append(task)
    return tasks


# Process every image in INPUT_DIR (commented out by default — uncomment to run).
# for img_path in sorted(p for p in Path(INPUT_DIR).iterdir()
#                        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}):
#     n_t, n_d = tile_and_save(img_path, model, OUT_ROOT)
#     print(f"  -> {n_t} tiles, {n_d} detections")
#
# tasks = build_ls_tasks(OUT_ROOT, [ID_TO_NAME[i] for i in range(NUM_CLASSES)], URL_PREFIX)
# (OUT_ROOT / "tasks.json").write_text(json.dumps(tasks, indent=2))
# print(f"Wrote {OUT_ROOT / 'tasks.json'} — {len(tasks)} tasks "
#       f"({sum(len(t.get('predictions', [{'result': []}])[0]['result']) for t in tasks)} boxes)")

---
## Where this lives in the codebase

The notebook reproduces every step *inline* so it can be read top-to-bottom, but the same logic is factored into the importable package under `src/`:

| Notebook section | Module | Purpose |
|---|---|---|
| Class metadata | `src/config.py` | single source of truth for the 9-class system, hex colours, palettes, default hyperparameters |
| Step 1 — synthetic data generation | `src/data_generation.py` | per-class procedural drawers, compositing (`paste_symbol`, `obb_to_yolo`), layout generators, photographic augmentation, `generate_dataset()`, `mix_real_data()` |
| Step 2 — training & evaluation | `src/training.py` | `train_local()` / `train_remote()`, `evaluate()` (confusion matrix + per-class metrics), `plot_evaluation()` |
| Step 3 — adaptive tiled inference | `src/inference.py` | `Detection`, `TileInfo`, `tile_starts`, `predict_tiled`, `estimate_stitch_size`, `predict_adaptive`, `predict_adaptive_with_tiles`, NMS primitives |
| Step 3 — rendering | `src/rendering.py` | `draw_svg_icon`, `render_two_panel`, `render_scheme`, `render_tile_grid`, `render_tiles_panel`, `render_reassembly`, `detections_to_svg`, adaptive `compute_stroke_width` |
| Step 3 — Label Studio export | `src/label_studio.py` | `tile_and_save`, `build_tasks_json`, `obb_corners_to_rotated_rect` |
| Pipeline orchestration | `src/pipeline.py` | `classify_stitches`, `run_pipeline`, `detections_to_json` |
| Upstream image generation | `src/generation.py` | `generate_image()` — Google Gemini wrapper |

The user-facing surfaces sit at the project root:

* **`main.py`** — argparse-based CLI with one subcommand per pipeline stage (`generate`, `train`, `evaluate`, `infer`, `tile`, `pipeline`).
* **`app.py`** — Streamlit application: pick an example image, generate one with Gemini, or upload your own → adaptive tiled inference → reconstructed scheme → "Inspect tiling" panel showing the split and the post-NMS reassembly.

That is the whole pipeline: synthetic data generation, training on a real-plus-synthetic mix, and adaptive tiled inference with rotated boxes. Step 3's predictions feed directly back into Step 1 via Label Studio — the human corrections become new training labels, which is what keeps the model improving over time.